In [1]:
import os
import shutil

# 1. Define the PyTorch checkpoint directory path
cache_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints')

# 2. Clear out the corrupted download if it exists
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("🧹 Corrupted cache cleared successfully!")
else:
    print("Cache directory was already clear or not found.")

Cache directory was already clear or not found.


In [2]:
import os
import h5py
import io
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
class HAM10000Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        
        # Load standard JPEG image safely
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx]['label'])

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

# 1. Custom Dataset Class matching GroundTruth structure
class HAM10000Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Your CSV row column is named 'image' (e.g. "ISIC_0024306")
        img_id = self.df.iloc[idx]['image']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx]['label_idx'])

        if self.transform:
            image = self.transform(image)

        return image, label

if __name__ == '__main__':
    # 2. Paths Configuration
    DATASET_DIR = "/Users/aditya/Downloads/archive (5)"
    IMG_DIR = os.path.join(DATASET_DIR, "all_images")
    CSV_PATH = os.path.join(DATASET_DIR, "GroundTruth.csv") # Points to your uploaded file

    print("🚀 Loading and parsing GroundTruth one-hot columns...")
    df = pd.read_csv(CSV_PATH)
    
    # Target condition classes tracked in order
    class_columns = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
    
    # Create a string label category column based on the column that equals 1.0
    df['label'] = df[class_columns].idxmax(axis=1)
    
    # Convert those label strings directly into numerical indices (0 to 6)
    df['label_idx'] = np.argmax(df[class_columns].values, axis=1)
    
    print(f"✅ Successfully mapped categories! Class Index Mapping: {class_columns}")

    # Split into train/validation sets cleanly using stratification
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_idx'])

    # 3. Image Augmentation Configurations
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = HAM10000Dataset(train_df, IMG_DIR, transform=transform)
    val_dataset = HAM10000Dataset(val_df, IMG_DIR, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

    # 4. Initialize Neural Net Model 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using compute unit target: {device}")
    
    model = models.mobilenet_v3_small(weights=None) 
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, 7)
    model = model.to(device)

    # 5. Optimization Loop parameters
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print("🏋️ Initiating Network Training Iterations...")
    model.train()
    for epoch in range(3):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f"Epoch {epoch+1} finished. Target cross loss: {running_loss/len(train_loader):.4f}")

    torch.save(model.state_dict(), 'ham10000_skin_model.pth')
    print("🎉 Saved weight models safely to 'ham10000_skin_model.pth'")